# Generating text using GPT-2

```Thank you Tel Aviv. Great Tel Aviv, amazing Tel Aviv. This is an amazing exercise, tremendous exercise. The best exercise for the AI COE.```

```But I'm tired. I'm tired writing my speeches, great speeches, amazing speeches, amazing people. Create a model, tremendous model, to write my speech.```

```~Trump```

(Well it's actually Cordova...)

#### Imports

In [2]:
# pytorch
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
# Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# other
import os
from time import time

c:\Users\yovel\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Generate text from a pre-trained model

```First, generate text using some great decoding model, like GPT-2, but not BERT, BERT can't decode, the democrats in google made it. Decode using beam search, Top-k and Top-p, great methods, amazing methods, great America. Implement this methods by yourself, don't use the ones in the transformers library, the chinese wrote them.```

```I heard the radical democrats are using AutoTokenizer and AutoModelForCausalLM from transformers. They think they can win with this, they can't. They can't.```

In [3]:
input_text = 'Look at me! I am Trump!'
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4493.06it/s]


In [ ]:
def top_k_decode(probs, k):
    top_k = probs.topk(k, dim=1)
    top_k_probs = top_k.values
    top_k_tokens = top_k.indices
    token_index = torch.multinomial(top_k_probs, 1)
    return top_k_tokens.gather(1, token_index)

def top_k_decoder(k):
    return lambda probs: top_k_decode(probs, k)

def top_p_decode(probs, p):
    sorted = probs.sort(dim=1)
    sorted_probs = sorted.values
    sorted_indices = sorted.indices 
    sorted_probs[sorted_probs.cumsum(dim=1) < p] = 0
    sorted_probs = sorted_probs/torch.sum(sorted_probs)
    token_index = torch.multinomial(sorted_probs, 1)
    return sorted_indices.gather(1, token_index)

def top_p_decoder(p):
    return lambda probs: top_p_decode(probs, p)


def decoding(model, tokenizer, input_text, n_tokens, decoder):
    inputs = tokenizer.encode(input_text, return_tensors="pt")
    with torch.no_grad():
        for i in range(n_tokens):
            outputs = model(inputs)
            preds = outputs.logits
            probs = torch.softmax(preds[:, -1, :], dim=-1)
            token = decoder(probs)        
            inputs = torch.cat([inputs, token], dim=1)
            if token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(inputs[0], skip_special_tokens=True)

def beam_search(model, tokenizer, input_text, n_tokens, N):
    inputs = tokenizer.encode(input_text, return_tensors="pt")
    options = [(inputs, 1)]
    with torch.no_grad():
        for j in range(n_tokens):
            new_options = [] 
            for option, prob in options:
                outputs = model(option)
                preds = outputs.logits
                top_tokens = torch.softmax(preds[:, -1, :], dim=-1).topk(N, dim=1)
                top_indices = top_tokens.indices
                top_probs = top_tokens.values
                for i in range(N):
                    new_option = torch.cat([option, top_indices[0][i].unsqueeze(0).unsqueeze(0)], dim=1)
                    new_prob = prob * top_probs[0][i]
                    new_options.append((new_option, new_prob))
            options = sorted(new_options, key=lambda x: x[1], reverse=True)[:N]
        
            if any(option[:, -1].item() == tokenizer.eos_token_id for option, prob in options):
                break
    winner = options[0][0]
    return tokenizer.decode(winner[0], skip_special_tokens=True)
                                   
            

k = 20
p = 0.75
print(decoding(model, tokenizer, input_text, 100, top_k_decoder(k)))
print(decoding(model, tokenizer, input_text, 100, top_p_decoder(p)))
print(beam_search(model, tokenizer, input_text, 100, 25))

Look at me! I am Trump! — Donald J. Trump


#### Read the data

```Now create a dataset with my speeches from ```https://www.kaggle.com/datasets/christianlillelund/donald-trumps-rallies.
```, amazing speeches, great speeches, great writing. Thank you, letters, you're heroes.```

```Decide a size for the input sequence length, and divide my amazing speeches to sequences of this size. The democrats are using 128, but we're at least double of them! We're double in size! Double in patriotism! Great double.```

In [56]:
input_length = 256
speeches = [] 
for f in os.listdir("archive"):
    speeches.append(open(f"archive/{f}", "r", encoding="utf-8").read())
data = '\n'.join(speeches)

'''
inputs = []
for i in range(np.ceil(len(data)/input_length).astype(int)):
    inputs.append(data[input_length*i:input_length*(i+1)])
'''
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.encode(data, truncation=True, padding="max_length", max_length=input_length, return_tensors="pt", return_overflowing_tokens=True)

#### Tokenize

```Use the tremendous method encode, to encode the great data. Only the USA has such data.```

In [57]:
class Dataset(TensorDataset):
    def __init__(self, inputs):
        self.inputs = inputs

    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx): 
        input_ids = self.inputs[idx].squeeze(0)
        return {
            "input_ids": input_ids,
        }

train, test = train_test_split(inputs, test_size=0.3)



#### Training

```Now, fine-tune your amazing model with the amazing dataset, and generate my next great speech. It should be a tremendous speech, amazing speech, hero speech. A speech you cry from, if you're not a man, men don't cry, only democrats.```

In [ ]:
training_args = TrainingArguments(output_dir="gpt2-finetuned",
                                  num_train_epochs=5,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=4,
    per_device_eval_batch_size=4)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=Dataset(train),
    eval_dataset=Dataset(test),
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

c:\Users\yovel\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# Training done in colab
model_trained = AutoModelForCausalLM.from_pretrained("gpt-trump4")
tokenizer = AutoTokenizer.from_pretrained("gpt-trump4")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7030.49it/s]


In [12]:
print(decoding(model_trained, tokenizer, input_text, 100, top_k_decoder(6)))

Look at me! I am Trump! You know that! I'm the one that beat him in the debate! I'm the one that beat them in the debate. You know that too. No, no, no, no, it's okay, I just want to say hello. Thank you. Thank you. Thank you. Thank you. I'm delighted to be in Manchester with thousands of hardworking patriots who believe in God, family, and country. We stand on the shoulders of American heroes who crossed the oceans, settled


Sample output:
Look at me! I am Trump! I am Trump! And I am so proud of you. We are going to win Michigan, and we're going to win the great state of Ohio. We're going to win it, and we're going to win the great state of Wisconsin, and I'm so proud to have won that state. We're going to win it, and we're going to win the great state of North Carolina. We're going to win it, and we're going to win it very, very easily, very quickly. And we are so proud of you, because you've done an incredible job. And you've shown the world that you can't let that happen to your state. You've shown the world that you can't let that happen to our country. We're going to win North Carolina, we're going to win North Carolina, and we are going to win the great state of Ohio. And we are so proud of all those wonderful patriots that came together in the great state of Ohio and we're thrilled tonight to be joined by many more patriots. I want to start with the people of Ohio, and they've been tremendous. They've been incredible and they love you. They love you. They love you. And they want to thank the incredible state of Wisconsin for giving us the chance, and we are so thrilled to have it, and I'm thrilled to have it going on. We're thrilled. We're thrilled.